In [ ]:
import os, sys, glob, subprocess
from pathlib import Path

if os.path.exists("/kaggle/input"):
    REPO = Path("/kaggle/working/Human_Action_Recognition-mlops")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/s-gasaro/Human_Action_Recognition-mlops", str(REPO)], check=True)
    csv = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)[0]
    IMAGE_DIR = Path(csv).parent / "train"
    ON_KAGGLE = True
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    IMAGE_DIR = REPO / "data" / "train"
    ON_KAGGLE = False

sys.path.insert(0, str(REPO / "src"))
SPLITS_CSV = REPO / "data" / "splits.csv"
MODELS_DIR = REPO / "models"
print(REPO, IMAGE_DIR, ON_KAGGLE)

## 1. Data loading
Load the stratified 70/15/15 split created by `src/preprocessing.py`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from PIL import Image
import preprocessing as pp
import model as hm

df = pp.load_splits(SPLITS_CSV)
class_names = pp.get_class_names(df)
train_df, val_df, test_df = [df[df.split == s] for s in ("train", "val", "test")]
print(len(train_df), len(val_df), len(test_df), len(class_names))
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. EDA

### Class distribution

In [ ]:
df.groupby(["label", "split"]).size().unstack().plot(kind="bar", stacked=True, figsize=(12, 4))
plt.ylabel("images"); plt.tight_layout(); plt.show()

### Sample image per class

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, c in zip(axes.ravel(), class_names):
    f = train_df[train_df.label == c].sample(1, random_state=0).filename.iloc[0]
    ax.imshow(Image.open(IMAGE_DIR / f)); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

### Image size and aspect ratio (random sample of 1000)

In [ ]:
sample = train_df.sample(1000, random_state=0)
sizes = np.array([Image.open(IMAGE_DIR / f).size for f in sample.filename])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(sizes[:, 0], bins=30, alpha=.6, label="width"); ax[0].hist(sizes[:, 1], bins=30, alpha=.6, label="height"); ax[0].legend()
ax[1].hist(sizes[:, 0] / sizes[:, 1], bins=30); ax[1].set_title("aspect ratio (w/h)")
plt.tight_layout(); plt.show()

### Brightness per class (20 images per class, mean gray level)

In [ ]:
rows = []
for c in class_names:
    for f in train_df[train_df.label == c].sample(20, random_state=0).filename:
        rows.append((c, np.asarray(Image.open(IMAGE_DIR / f).convert("L").resize((64, 64))).mean()))
b = pd.DataFrame(rows, columns=["label", "brightness"])
b.boxplot(by="label", column="brightness", figsize=(12, 4), rot=60); plt.suptitle(""); plt.ylabel("brightness"); plt.tight_layout(); plt.show()

## 3. Preprocessing and augmentation
Images are resized to 224x224 and scaled with MobileNetV2 `preprocess_input`. Augmentation (flip, rotation, zoom, contrast) is applied to training data only.

In [ ]:
aug = pp.augmentation()
f = train_df.filename.iloc[0]
img = tf.image.resize(tf.io.decode_image(tf.io.read_file(str(IMAGE_DIR / f)), channels=3, expand_animations=False), pp.IMG_SIZE)[None]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img[0].numpy().astype("uint8")); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(aug(img, training=True)[0].numpy().astype("uint8"))
for ax in axes: ax.axis("off")
plt.show()

make = lambda d, **k: pp.make_dataset(d, class_names, IMAGE_DIR, **k)
val_ds, test_ds = make(val_df), make(test_df)
train_plain = make(train_df, training=True, augment=False)
train_aug = make(train_df, training=True, augment=True)

## 4. Experiments
Models are compared on the validation set. All use MobileNetV2 (ImageNet weights) with GAP, Dropout and a softmax head.

In [ ]:
results, models = {}, {}
def log(name, m):
    results[name] = hm.evaluate(m, val_ds); models[name] = m
    print(name, results[name])

### Exp 1: frozen base, no augmentation, no dropout

In [ ]:
m1 = hm.build_model(len(class_names), dropout=0.0, lr=1e-3)
hm.train(m1, train_plain, val_ds, epochs=8)
log("exp1_frozen", m1)

### Exp 2: frozen base + augmentation + dropout 0.3

In [ ]:
m2 = hm.build_model(len(class_names), dropout=0.3, lr=1e-3)
hm.train(m2, train_aug, val_ds, epochs=10)
log("exp2_aug_dropout_lr1e-3", m2)

### Exp 3: exp 2 + fine-tune top 30 base layers, low LR (1e-5), early stopping

In [ ]:
m3 = tf.keras.models.clone_model(m2)
m3.set_weights(m2.get_weights())
hm.unfreeze(m3, 30, lr=1e-5)
hm.train(m3, train_aug, val_ds, epochs=8)
log("exp3_finetune", m3)

### Exp 4: learning rate 1e-3 vs 1e-4
Exp 2's configuration (aug + dropout) with LR 1e-4; the 1e-3 run is exp 2.

In [ ]:
m4 = hm.build_model(len(class_names), dropout=0.3, lr=1e-4)
hm.train(m4, train_aug, val_ds, epochs=10)
log("exp4_aug_dropout_lr1e-4", m4)

## 5. Comparison (validation set)

In [ ]:
table = pd.DataFrame(results).T[["accuracy", "loss", "precision", "recall", "f1"]]
display(table.round(4))
best_name = table["f1"].idxmax()
best = models[best_name]
print("Best:", best_name)

## 6. Best model on the test set

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
test_metrics = hm.evaluate(best, test_ds)
print(test_metrics)
y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds])
y_pred = best.predict(test_ds, verbose=0).argmax(1)
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap="Blues"); ax.set_xticks(range(len(class_names)), class_names, rotation=90); ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.tight_layout(); plt.show()

## 7. Prediction demo on 3 test images

In [ ]:
def predict(model, image_path, class_names, top_k=3):
    img = tf.image.resize(tf.io.decode_image(tf.io.read_file(str(image_path)), channels=3, expand_animations=False), pp.IMG_SIZE)
    probs = model.predict(pp.preprocess_image(img)[None], verbose=0)[0]
    return [(class_names[i], float(probs[i])) for i in probs.argsort()[::-1][:top_k]]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, r) in zip(axes, test_df.sample(3, random_state=1).iterrows()):
    top = predict(best, IMAGE_DIR / r.filename, class_names)
    ax.imshow(Image.open(IMAGE_DIR / r.filename)); ax.axis("off")
    ax.set_title(f"true: {r.label}\npred: {top[0][0]} ({top[0][1]:.2f})")
plt.tight_layout(); plt.show()

## 8. Save the best model

In [ ]:
meta = {"experiment": best_name, **test_metrics}
hm.save_model(best, class_names, meta, MODELS_DIR)
if ON_KAGGLE:
    hm.save_model(best, class_names, meta, "/kaggle/working/models")
print(os.listdir(MODELS_DIR))